# End-to-End NLP Deep Learning Pipeline

### Task: Text Classification (Sentiment Analysis) using DistilBERT

This notebook demonstrates an end-to-end deep learning pipeline for NLP, from data loading to model saving. We use `DistilBERT`, a distilled, faster version of BERT that achieves 95% of BERT's performance while keeping the resource footprint student-friendly.

## Step 1: Install and Import Dependencies

First, we need to install the required Hugging Face and PyTorch ecosystems.

In [ ]:
!pip install -q transformers datasets accelerate evaluate

import os
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_scheduler
from tqdm.auto import tqdm

## Step 2: Environment and Device Setup

Ensure reproducibility and detect if a GPU (CUDA) or Apple Silicon (MPS) is available to accelerate training.

In [ ]:
# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Select hardware accelerator
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Step 3: Load the Dataset

We will use the classic **IMDb movie review dataset**. For a student portfolio pipeline, we will downsample it to keep the notebook runtime fast and predictable.

In [ ]:
print("Loading IMDb dataset...")
raw_datasets = load_dataset("imdb")

# Downsample to 2,000 train and 500 test samples for quick execution
train_size = 2000
test_size = 500

train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(train_size))
test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(test_size))

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Sample data point:\nText: {train_dataset[0]['text'][:150]}...\nLabel: {train_dataset[0]['label']} (0=Neg, 1=Pos)")

## Step 4: Data Preprocessing (Tokenization)

Deep learning models don't read text directly; they process numbers. We load the checkpoint's corresponding tokenizer to convert words into sequence tokens, handle padding, and truncation.

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    # Truncate to max 512 tokens (BERT's hard limit) and pad sequences
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Format datasets for PyTorch (keep only tensor-compatible columns)
columns_to_keep = ["input_ids", "attention_mask", "label"]
tokenized_train = tokenized_train.remove_columns([col for col in tokenized_train.column_names if col not in columns_to_keep])
tokenized_test = tokenized_test.remove_columns([col for col in tokenized_test.column_names if col not in columns_to_keep])

# Rename 'label' to 'labels' because HF models expect 'labels' as the key
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Convert to PyTorch tensors
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

## Step 5: DataLoaders and PyTorch Wrappers

Convert the Hugging Face datasets into native PyTorch `DataLoaders` to manage batching and shuffling automatically during training.

In [ ]:
BATCH_SIZE = 16

train_dataloader = DataLoader(tokenized_train, shuffle=True, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(tokenized_test, batch_size=BATCH_SIZE)

# Quick sanity check on a batch
batch = next(iter(train_dataloader))
print({k: v.shape for k, v in batch.items()})

## Step 6: Initialize Model, Optimizer, and Scheduler

We load the pretrained `DistilBERT` architecture with an added sequence classification head (2 output classes: Positive or Negative).

In [ ]:
# Initialize model with 2 output labels
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
model.to(device)

# Hyperparameters
EPOCHS = 3
LEARNING_RATE = 2e-5

# Optimizer and LR Scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

## Step 7: Training Loop

This is a native PyTorch training loop where we pass inputs to the model, compute the gradient loss, backpropagate, and step our optimizer.

In [ ]:
progress_bar = tqdm(range(num_training_steps), desc="Training")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_dataloader:
        # Move batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Optimization step
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        progress_bar.update(1)
        
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{EPOCHS} completed. Average Loss: {avg_loss:.4f}")

## Step 8: Evaluation Pipeline

We freeze the model parameters using `model.eval()` and evaluate the model's accuracy on our testing subset using the Hugging Face `evaluate` module.

In [ ]:
metric = evaluate.load("accuracy")
model.eval()

for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    
    with torch.no_grad():
        outputs = model(**batch)
        
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    
    # Accumulate evaluation metrics
    metric.add_batch(predictions=predictions, references=batch["labels"])

eval_results = metric.compute()
print(f"\nFinal Test Accuracy: {eval_results['accuracy'] * 100:.2f}%")

## Step 9: Save the Model and Tokenizer Artifacts

To make this a complete pipeline, we save both the fine-tuned model and its specific tokenizer weights to a local directory so it can be committed, cached, or served later.

In [ ]:
OUTPUT_DIR = "./saved_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save using native Hugging Face serialization
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model and Tokenizer successfully saved to '{OUTPUT_DIR}'!")

## Step 10: Model Inference Production Testing

Finally, let's load our saved model from disk and test it against completely custom text strings to verify it works in a simulated deployment scenario.

In [ ]:
print("\n--- Testing Model Inference Pipeline ---")

# Load saved artifacts
loaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
loaded_model.eval()

def predict_sentiment(text: str):
    # Preprocess custom text input
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    
    with torch.no_grad():
        outputs = loaded_model(**inputs)
        
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    return sentiment, confidence

# Examples
sample_texts = [
    "This tutorial was absolutely phenomenal. Clear instructions and solid performance!",
    "I wasted two hours of my life on this piece of garbage movie. Don't watch it."
]

for text in sample_texts:
    sentiment, score = predict_sentiment(text)
    print(f"\nText: \"{text}\"")
    print(f"Predicted Sentiment: {sentiment} ({score*100:.2f}% Confidence)")